# CommissionLens
## Commission-Adjusted Alpha Prediction in Indian Mutual Funds

**Tech Stack:** Python · XGBoost · SHAP · scipy (XIRR) · Pandas · Matplotlib  
**Data:** AMFI NAV data · RBI DBIE macro · NSE Nifty 50 benchmark  
**Goal:** Predict whether a mutual fund will generate net-positive alpha after accounting for its commission (expense ratio gap between regular and direct plans).

---


## 0. Setup & Imports

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# Install dependencies if needed
import subprocess
for pkg in ['xgboost', 'shap', 'pyarrow', 'scipy']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run(['pip', 'install', pkg, '-q'])

import xgboost as xgb
import shap
from scipy.optimize import brentq
from sklearn.metrics import roc_auc_score, f1_score, mean_squared_error, r2_score, mean_absolute_error

# Paths — update DATA_ROOT if running locally with real data
import os
DATA_ROOT = os.path.dirname(os.getcwd()) if os.path.exists('../data') else '.'
RAW_DIR   = f'{DATA_ROOT}/data/raw'
PROC_DIR  = f'{DATA_ROOT}/data/processed'
REP_DIR   = f'{DATA_ROOT}/reports'
for d in [RAW_DIR, PROC_DIR, REP_DIR]:
    os.makedirs(d, exist_ok=True)

STYLE = {
    'bg':    '#0a0a0a', 'panel': '#111111', 'gold':  '#d4a017',
    'green': '#39ff14', 'red':   '#ff4444', 'text':  '#e8e8e8', 'grid': '#1f1f1f',
}
plt.rcParams.update({'figure.facecolor': STYLE['bg'], 'axes.facecolor': STYLE['panel'],
                     'text.color': STYLE['text'], 'axes.labelcolor': STYLE['text'],
                     'xtick.color': STYLE['text'], 'ytick.color': STYLE['text']})
print("Setup complete")


## 1. Data Pipeline

> **Note:** This notebook uses realistic synthetic data that mirrors Indian equity MF market structure.  
> To use real data, replace `generate_*` calls with the real API fetchers in `src/fetch_real_data.py`.


In [ ]:

# Run the data generator (if data doesn't exist yet)
import sys
sys.path.insert(0, '../src')

if not os.path.exists(f'{RAW_DIR}/fund_pairs.csv'):
    exec(open('../src/generate_synthetic_data.py').read())
else:
    print("Raw data already exists, loading...")

fund_pairs = pd.read_csv(f'{RAW_DIR}/fund_pairs.csv')
benchmark  = pd.read_csv(f'{RAW_DIR}/benchmark_nifty50.csv', parse_dates=['date'])
macro      = pd.read_csv(f'{RAW_DIR}/macro_data.csv', parse_dates=['date'])
nav_df     = pd.read_parquet(f'{RAW_DIR}/nav_histories.parquet')
nav_df['date'] = pd.to_datetime(nav_df['date'])

print(f"Fund pairs : {fund_pairs.shape}")
print(f"Benchmark  : {benchmark.shape}")
print(f"Macro data : {macro.shape}")
print(f"NAV history: {nav_df.shape}")
fund_pairs.head()


### 1b. Exploratory Data Analysis

In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 8), facecolor=STYLE['bg'])
axes = axes.flatten()

# Expense gap distribution
axes[0].hist(fund_pairs['expense_gap'], bins=30, color=STYLE['gold'], edgecolor=STYLE['bg'])
axes[0].set_title('Expense Gap (Reg - Dir)', color=STYLE['gold'])
axes[0].set_xlabel('Expense Gap (%)')

# AUM distribution
axes[1].hist(np.log1p(fund_pairs['aum_cr']), bins=30, color='#4fc3f7', edgecolor=STYLE['bg'])
axes[1].set_title('AUM Distribution (log)', color=STYLE['gold'])
axes[1].set_xlabel('log(AUM ₹Cr)')

# Beta distribution
axes[2].hist(fund_pairs['beta'], bins=30, color='#ab47bc', edgecolor=STYLE['bg'])
axes[2].set_title('Fund Beta Distribution', color=STYLE['gold'])

# Nifty 50 price history
axes[3].plot(benchmark['date'], benchmark['nifty50_close']/1e3, color=STYLE['green'], lw=1.5)
axes[3].set_title('Nifty 50 (2018–2024)', color=STYLE['gold'])
axes[3].set_ylabel('Index (000s)')

# Repo rate
axes[4].plot(macro['date'], macro['repo_rate'], color=STYLE['red'], lw=2)
axes[4].set_title('RBI Repo Rate', color=STYLE['gold'])
axes[4].set_ylabel('%')

# CPI
axes[5].plot(macro['date'], macro['cpi'], color=STYLE['gold'], lw=2)
axes[5].fill_between(macro['date'], macro['cpi'], alpha=0.2, color=STYLE['gold'])
axes[5].set_title('CPI Inflation', color=STYLE['gold'])
axes[5].set_ylabel('%')

for ax in axes:
    ax.set_facecolor(STYLE['panel'])
    ax.spines[:].set_color(STYLE['gold'])
    ax.grid(color=STYLE['grid'], lw=0.5)
plt.suptitle('CommissionLens — EDA', color=STYLE['gold'], fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Feature Engineering

In [ ]:

if not os.path.exists(f'{PROC_DIR}/features.parquet'):
    exec(open('../src/feature_engineering.py').read())
    features = build_features()
else:
    features = pd.read_parquet(f'{PROC_DIR}/features.parquet')
    features['quarter'] = pd.to_datetime(features['quarter'])
    print(f"Features loaded: {features.shape}")

print("\nClass balance:")
print(features['commission_justified'].value_counts(normalize=True).round(3))
features.describe().T[['mean','std','min','max']].round(3)


In [ ]:

# Feature correlation heatmap
import matplotlib.patches as mpatches

FEAT_COLS = [
    'expense_gap','aum_log','manager_tenure_yrs','portfolio_turnover','beta_static',
    'rolling_sharpe_4q','rolling_sharpe_8q','information_ratio_4q','alpha_persistence',
    'gross_alpha_q','repo_rate','cpi','yield_slope','fii_dii_ratio'
]
corr = features[FEAT_COLS + ['net_alpha_next_q']].dropna().corr()

fig, ax = plt.subplots(figsize=(14, 10), facecolor=STYLE['bg'])
ax.set_facecolor(STYLE['panel'])
im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=8, color=STYLE['text'])
ax.set_yticklabels(corr.columns, fontsize=8, color=STYLE['text'])
ax.set_title('Feature Correlation Matrix', color=STYLE['gold'], fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


## 3. Walk-Forward Cross-Validation

In [ ]:

if not os.path.exists(f'{PROC_DIR}/walkforward_predictions.parquet'):
    exec(open('../src/walk_forward_cv.py').read())
    pred_df, metrics_df = run_walk_forward()
else:
    pred_df    = pd.read_parquet(f'{PROC_DIR}/walkforward_predictions.parquet')
    metrics_df = pd.read_csv(f'{PROC_DIR}/walkforward_metrics.csv', parse_dates=['quarter'])
    print(f"Predictions: {pred_df.shape}, Metrics: {metrics_df.shape}")

pred_df['quarter'] = pd.to_datetime(pred_df['quarter'])
metrics_df.head(10)


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=STYLE['bg'])
for ax in axes:
    ax.set_facecolor(STYLE['panel'])
    ax.spines[:].set_color(STYLE['gold'])
    ax.grid(color=STYLE['grid'], lw=0.5)

axes[0].plot(metrics_df['quarter'], metrics_df['auc'], color=STYLE['gold'], lw=2, marker='o', ms=4)
axes[0].axhline(0.5, color=STYLE['red'], ls='--', lw=1, label='Random baseline')
axes[0].set_title('AUC-ROC over time', color=STYLE['gold'])
axes[0].set_ylabel('AUC'); axes[0].legend(labelcolor=STYLE['text'], facecolor=STYLE['panel'])

axes[1].plot(metrics_df['quarter'], metrics_df['f1'],   color='#4fc3f7', lw=2, marker='s', ms=4)
axes[1].set_title('F1 Score over time', color=STYLE['gold']); axes[1].set_ylabel('F1')

axes[2].plot(metrics_df['quarter'], metrics_df['rmse'], color='#ab47bc', lw=2, marker='^', ms=4)
axes[2].set_title('RMSE (net alpha) over time', color=STYLE['gold']); axes[2].set_ylabel('RMSE')

plt.suptitle('Walk-Forward CV Performance', color=STYLE['gold'], fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print(f"Mean AUC : {metrics_df['auc'].mean():.4f}")
print(f"Mean F1  : {metrics_df['f1'].mean():.4f}")
print(f"Mean RMSE: {metrics_df['rmse'].mean():.4f}")


## 4. SHAP Explainability

In [ ]:

# Retrain final model on full train set for SHAP
FEATURE_COLS = [
    'expense_gap','direct_er','regular_er','aum_log','manager_tenure_yrs',
    'portfolio_turnover','beta_static','rolling_sharpe_4q','rolling_sharpe_8q',
    'rolling_beta','information_ratio_4q','alpha_persistence','gross_alpha_q',
    'net_alpha_q','repo_rate','cpi','yield_slope','fii_flow_cr_qtly',
    'dii_flow_cr_qtly','fii_dii_ratio'
]
FEATURE_LABELS = {
    'expense_gap':'Expense Ratio Gap', 'aum_log':'AUM (log)', 'beta_static':'Beta',
    'rolling_sharpe_4q':'Sharpe (4Q)', 'rolling_sharpe_8q':'Sharpe (8Q)',
    'information_ratio_4q':'Info Ratio (4Q)', 'alpha_persistence':'Alpha Persistence',
    'gross_alpha_q':'Gross Alpha (Q)', 'net_alpha_q':'Net Alpha (Q)',
    'repo_rate':'Repo Rate', 'cpi':'CPI', 'yield_slope':'Yield Slope',
    'fii_flow_cr_qtly':'FII Flows', 'dii_flow_cr_qtly':'DII Flows',
    'fii_dii_ratio':'FII/DII Ratio', 'rolling_beta':'Rolling Beta',
    'manager_tenure_yrs':'Manager Tenure', 'portfolio_turnover':'Port. Turnover',
    'direct_er':'Direct ER', 'regular_er':'Regular ER'
}

df = features.copy()
for col in FEATURE_COLS:
    df[col] = df[col].fillna(df[col].median())

TRAIN_CUTOFF = pd.Timestamp('2022-12-31')
train = df[df['quarter'] <= TRAIN_CUTOFF]
test  = df[df['quarter'] >  TRAIN_CUTOFF]

X_train = train[FEATURE_COLS].values
X_test  = test[FEATURE_COLS].values
y_cls   = train['commission_justified'].values

scale_pos = (y_cls == 0).sum() / max(1, (y_cls == 1).sum())
cls_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, scale_pos_weight=scale_pos,
    use_label_encoder=False, eval_metric='logloss', random_state=42, verbosity=0
)
cls_model.fit(X_train, y_cls)
print(" Model retrained for SHAP analysis")


In [ ]:

# SHAP values
sample_idx = np.random.choice(len(X_test), min(300, len(X_test)), replace=False)
X_sample   = X_test[sample_idx]

explainer   = shap.TreeExplainer(cls_model)
shap_values = explainer(X_sample)
mean_abs    = np.abs(shap_values.values).mean(axis=0)
order       = np.argsort(mean_abs)[::-1][:15]

labels = [FEATURE_LABELS.get(FEATURE_COLS[i], FEATURE_COLS[i]) for i in order]
vals   = mean_abs[order]

fig, ax = plt.subplots(figsize=(12, 7), facecolor=STYLE['bg'])
ax.set_facecolor(STYLE['panel']); ax.spines[:].set_color(STYLE['gold'])
ax.tick_params(colors=STYLE['text'])
colors = [STYLE['green'] if v > vals.mean() else '#555555' for v in vals[::-1]]
ax.barh(labels[::-1], vals[::-1], color=colors)
ax.set_xlabel('Mean |SHAP value|', color=STYLE['text'])
ax.set_title('SHAP Feature Importance — Commission Justified', color=STYLE['gold'],
             fontsize=13, fontweight='bold')
ax.grid(axis='x', color=STYLE['grid'], lw=0.5)
plt.tight_layout(); plt.show()


In [ ]:

# SHAP dependence plots — top 6 features
top6_idx = order[:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 8), facecolor=STYLE['bg'])
axes = axes.flatten()
for k, feat_i in enumerate(top6_idx):
    ax = axes[k]
    ax.set_facecolor(STYLE['panel']); ax.spines[:].set_color(STYLE['gold'])
    ax.tick_params(colors=STYLE['text'], labelsize=8)
    label = FEATURE_LABELS.get(FEATURE_COLS[feat_i], FEATURE_COLS[feat_i])
    sc = ax.scatter(X_sample[:, feat_i], shap_values.values[:, feat_i],
                    c=X_sample[:, feat_i], cmap='RdYlGn', alpha=0.5, s=12)
    ax.axhline(0, color=STYLE['gold'], lw=0.8, ls='--')
    ax.set_title(label, color=STYLE['text'], fontsize=9)
    ax.set_xlabel('Feature Value', color=STYLE['text'], fontsize=8)
    ax.set_ylabel('SHAP', color=STYLE['text'], fontsize=8)
plt.suptitle('SHAP Dependence Plots (Top 6 Features)', color=STYLE['gold'],
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 5. SIP Back-Validation (2019–2023)

Comparing three strategies:
- **Strategy A (Model-Guided):** Invest in top-10% funds predicted to justify their commission each quarter
- **Strategy B (Naive Regular):** Equal-weight SIP across all 250 regular plans
- **Strategy C (Direct Benchmark):** Equal-weight SIP across all 250 direct plans (no commission counterfactual)


In [ ]:

# Load SIP results (pre-computed)
summary_txt = open(f'{REP_DIR}/sip_backtest_summary.txt').read()
print(summary_txt)


In [ ]:

# Load and display the charts
from IPython.display import Image, display
display(Image(f'{REP_DIR}/sip_backtest_chart.png'))


In [ ]:

display(Image(f'{REP_DIR}/xirr_comparison.png'))


In [ ]:

display(Image(f'{REP_DIR}/shap_summary_classification.png'))


## 6. Conclusions & Findings

### Key Results

| Metric | Value |
|---|---|
| Walk-Forward AUC (mean) | ~0.53 |
| Model-Guided XIRR | ~20.2% |
| Naive Regular XIRR | ~18.7% |
| Direct Benchmark XIRR | ~19.9% |
| Model vs Naive Δ XIRR | +1.5% |

### Interpretations

1. **Alpha is hard to predict** — AUC ~0.53 is consistent with semi-strong market efficiency. The value is in *regime-conditional ranking*, not precise prediction.

2. **Macro regime dominates** — Repo rate, CPI, and yield curve slope are the most informative features. Funds justify commissions more in falling-rate / low-volatility regimes.

3. **Alpha persistence matters** — The AR(1) coefficient of quarterly alpha and 4Q Sharpe ratio are the strongest fund-level predictors. Past alpha has mild but real predictive power.

4. **Commission cost is real** — Direct plans deliver +1.21% XIRR vs naive regular over 5 years on ₹5,000/month SIP. On a ₹50L corpus over 20 years, this compounds to ₹8-12L difference.

5. **Model adds value** — Model-guided selection of regular plans (+1.52% vs naive) actually *beats* even the direct benchmark, demonstrating that commission cost can be justified by better fund selection.

### Limitations & Future Work
- Real data would improve feature quality (actual expense gaps from AMFI, FII/DII from SEBI)
- Add fund manager change events as a structural break feature
- Incorporate portfolio overlap / factor exposures (Fama-French India factors)
- Build a Streamlit dashboard for live fund scoring


In [ ]:

print("CommissionLens pipeline complete.")
print("All outputs saved to reports/ directory:")
import os
for f in sorted(os.listdir(REP_DIR)):
    size = os.path.getsize(f'{REP_DIR}/{f}')
    print(f"  {f:<45} {size/1024:.1f} KB")
